# 02 · MLP on MNIST —— 用多层感知机突破线性限制

**家族位置**：`01_Fundamentals_MLP` 第 2 个项目（前：`01_Perceptron`；后：`03_Training_Tricks_Ablation`）

**任务背景**：感知机只能画直线，MNIST 手写数字识别却需要弯弯曲曲的决策面。本项目第一次跑通 PyTorch 的**完整训练范式**：数据 → 模型 → 训练 → 评估 → 可视化，目标是把测试准确率做到 **98%+**（对 MLP 而言是合理基线，不用卷积也能达到）。

**学习目标**
1. 理解 MLP 如何突破感知机的线性天花板（XOR 难题的现场验证）
2. 掌握 PyTorch 训练范式的每个环节：Dataset/DataLoader → 模型 → 损失 → 优化 → 评估
3. 建立评估直觉：训练/测试曲线、混淆矩阵、错误样本分析
4. 观察**过拟合**的早期信号——为 03 项目的正则化消融埋伏笔

## 1. 原理：从感知机到 MLP

### 感知机为什么失败

感知机的假设空间只有超平面（直线）。XOR 的两类点在对角位置互相"包围"，任何直线最多分对约一半——这就是 01 项目里它卡在 56% 的原因。

### MLP 的解法：组合非线性切分

把多个神经元叠成两层：

```
x ──┐
    ├─→ 隐藏层（多个神经元，各学一条切分）─→ 非线性激活（ReLU）
    │
    └─→ …  →  输出层做逻辑组合（AND/OR）  →  最终决策
```

第一层学出多条"线性切分"（每根线分一个方向），第二层把切分结果**非线性组合**起来——XOR 就能被拼出来了。层数加深 + 激活函数非线性，等价于在放大**假设空间**：

$$h = \sigma(W_1 x + b_1), \qquad \hat{y} = \mathrm{softmax}(W_2 h + b_2)$$

### 为什么需要非线性激活

如果激活是线性的，多层线性仍等价于单层线性（矩阵连乘还是矩阵），叠加就没有意义。**非线性（ReLU 等）才是深度有效的根本原因**。

### 分类头与损失

输出层给每个类别一个 logit，配 `CrossEntropyLoss`（内部 = softmax + 交叉熵，数值稳定），标签用 `{0..9}` 的 int64。

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import load_mnist_torch, make_xor
from common.engine import fit
from common.models import MLP, Perceptron
from common.utils import count_params, plot_decision_boundary, set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", DEVICE)
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)

## 2. 数据：MNIST

MNIST 是手写数字识别"教科书数据集"：60,000 训练 + 10,000 测试，28×28 灰度图，10 类。预处理：展平为 784 维、除以 255 后按固定均值/方差标准化（`common/data.py::load_mnist_torch`，首次运行自动下载到 `../data/`）。

In [ ]:
DATA_ROOT = ROOT / "data"
Xtr, ytr, Xte, yte = load_mnist_torch(str(DATA_ROOT), flatten=True)
print("训练集:", Xtr.shape, "| 测试集:", Xte.shape)
print("类别分布(训练集):", np.bincount(ytr.numpy()).tolist())

fig, axes = plt.subplots(1, 10, figsize=(12, 1.5))
for i in range(10):
    idx = int(np.where(ytr.numpy() == i)[0][0])
    axes[i].imshow(Xtr[idx].reshape(28, 28), cmap="gray")
    axes[i].axis("off")
plt.suptitle("每个类别各取一个样本", fontsize=12)
plt.savefig(FIGS / "fig0_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. 模型与训练范式

### 模型：784 → 256 → 128 → 10（+ Dropout 0.2）

`common/models.py::MLP`：两层隐藏层，ReLU 激活，输出层直接给 logits。加了 `Dropout(0.2)`——它在训练时随机"关掉"20% 的神经元，防止模型死记训练样本（正则化的一种；03 项目会系统对比）。

### 范式（家族统一，`common/engine.py`）

`Dataset` 装数据 → `DataLoader` 按 batch 喂给模型 → 前向算 logits → `CrossEntropyLoss` → 反向传播 `backward()` → 优化器 `step()`。这些全部封装在 `fit()` 里：

```python
hist = fit(model, train_loader, test_loader, epochs=10, lr=1e-3)
```

> 说明：为教学简化，本项目直接用官方测试集做每轮监控（val）；严格做法应从训练集再划出独立验证集，避免拿测试集调参。

In [ ]:
model = MLP(in_dim=784, hidden_dims=(256, 128), out_dim=10, dropout=0.2)
print("结构:", model)
print(f"可训练参数量: {count_params(model):,}")

batch_size = 128
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=512, shuffle=False)

## 4. 训练

超参数：`Adam(lr=1e-3)`、batch 128、10 epochs。CPU 上约 1 分钟；固定了随机种子，结果可复现。

In [ ]:
hist = fit(model, train_loader, test_loader, epochs=10, lr=1e-3, device=DEVICE)
print(f"\n最终测试准确率: {hist['val_acc'][-1]:.2%}")

## 5. 结果解读：看曲线

训练 loss 持续下降是"拟合训练集"；关键看**验证曲线**（这里 = 测试集）。

- 若 val loss 也在降 → 还在学有用的模式
- 若 val 停滞甚至回升、train 还在降 → **过拟合信号**（本项目第 6~10 轮就有这个苗头）

Dropout 已经帮我们压住了大部分过拟合；如果把它去掉，val 回升会更明显——03 项目会做这个对照实验。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(hist["train_loss"], label="train", marker="o", ms=3)
axes[0].plot(hist["val_loss"], label="val", marker="o", ms=3)
axes[0].set_title("Loss 曲线"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(hist["train_acc"], label="train", marker="o", ms=3)
axes[1].plot(hist["val_acc"], label="val", marker="o", ms=3)
axes[1].set_title("Accuracy 曲线"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(FIGS / "fig1_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(Xte)
    pred = logits.argmax(1)

acc = (pred == yte).float().mean().item()
print(f"测试集准确率: {acc:.2%}")

### 混淆矩阵

看错误不是均匀分布的——哪两个数字最像？这就是模型的"盲区"。

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(yte.numpy(), pred.numpy())

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=7)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("预测"); ax.set_ylabel("真实"); ax.set_title("混淆矩阵")
plt.colorbar(im)
plt.savefig(FIGS / "fig2_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
flat = cm_off.ravel().argsort()[::-1][:5]
pairs = [(int(r), int(c), int(cm_off[r, c])) for idx in flat for r, c in [np.unravel_index(idx, cm.shape)]]
print("最常被看错的 5 对（真实→预测）:", pairs)

### 错误样本

挑几个模型答错的样本，看看是真的"鬼画符"，还是连人眼都容易搞混的写法。

In [ ]:
wrong = torch.where(pred != yte)[0]
print(f"测试集共答错 {len(wrong)} 个（错误率 {(1 - acc):.2%}）")

fig, axes = plt.subplots(2, 5, figsize=(11, 4.5))
for ax, idx in zip(axes.ravel(), wrong[:10]):
    ax.imshow(Xte[idx].reshape(28, 28), cmap="gray")
    ax.set_title(f"真:{yte[idx].item()} 预:{pred[idx].item()}", fontsize=10)
    ax.axis("off")
plt.suptitle("前 10 个错误样本", fontsize=12)
plt.savefig(FIGS / "fig3_errors.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. 回归验证：MLP 能解决 XOR 吗

回到 01 项目的"伤心地"——同样的 XOR 任务，换成小 MLP（2→32→32→2）。为避免过重噪声带来的"不可避免误差"干扰对比，这里用低噪声版。下面先让感知机在同一份数据上跑一次（看它的线性天花板），再让 MLP 跑，差距一目了然。

In [ ]:
# 与 01 项目对比：先让感知机在同一份数据上试一次（线性天花板），再交给 MLP
set_seed(0)
X, y, _ = make_xor(n=500, noise=0.05, seed=0)
Xn, yn = X.numpy(), np.where(y.numpy() == 0, -1, 1)

pct = Perceptron(lr=1.0, max_epochs=200, seed=0).fit(Xn, yn)
acc_pct = pct.score(Xn, yn)
print(f"感知机（线性假设空间）在 XOR 上: {acc_pct:.2%}")

xor_loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)
mlp_xor = MLP(in_dim=2, hidden_dims=(32, 32), out_dim=2, dropout=0.0)
hist_xor = fit(mlp_xor, xor_loader, xor_loader, epochs=50, lr=1e-2, verbose=False)
acc_xor = hist_xor["val_acc"][-1]
print(f"MLP（非线性假设空间）在 XOR 上: {acc_xor:.2%}")

plot_decision_boundary(
    mlp_xor, X.numpy(), y.numpy(),
    title=f"MLP 的 XOR 决策边界（准确率 {acc_xor:.2%}）",
    to_pm1=lambda m, g: np.where(
        m(torch.as_tensor(g, dtype=torch.float32)).argmax(1).numpy() == 1, 1, -1),
)
plt.savefig(FIGS / "fig4_xor_boundary.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. 误差分析与总结

### 本项目的关键观察

| 观察 | 含义 |
|---|---|
| 测试 98.14%（10 epoch + Dropout） | 全连接网络在 MNIST 上的合理基线；不做任何卷积也能到 98% |
| 混淆集中在 3/5、4/9、7/2、7/9 | 形状相近的数字是模型（也是人眼）的天然盲区 |
| 后 5 轮 val 曲线走平、train 还在涨 | 模型容量开始"记"训练数据 → 过拟合的前兆 → Dropout/正则化登场（03 项目） |
| XOR：感知机 46.8% → MLP 96.0%（实测，见第 6 节） | **非线性 + 隐藏层 = 更强的假设空间**，这是深度学习的第一性原理 |

### 局限与下一步

- MLP 把 28×28 图像**展平成 784 个像素**，完全无视"相邻像素构成笔画/形状"的结构——它没有**平移不变性**，也没有**局部感受野**
- 这就是为什么图像任务要交给 CNN：`02_CNN_Family`（`01_LeNet_MNIST` 起步，可与我们现在的 98.14% 直接对比）
- 想在 01 家族内继续深挖？下一步 `03_Training_Tricks_Ablation`：同一 MLP 上对比 SGD/Adam/AdamW、有无 BN/Dropout，亲眼看到这些"trick"每个值多少分